# 🚬 흡연 분류 AI 해커톤 - V3 (최종 제출용)

**⭐ 핵심 개선사항:**
- Accuracy 기준 튜닝 (ROC-AUC ❌)
- 임계값 0.01 단위 세밀 탐색
- 클래스 가중치 balanced 적용
- 10개 시드 앙상블
- 피처 엔지니어링 대폭 강화

---

## 📌 STEP 1: 환경 설정

In [ ]:
# 1-1. 라이브러리 설치
!pip install -q xgboost lightgbm catboost

In [ ]:
# 1-2. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1-3. 경로 설정 (본인 경로에 맞게 수정!)
base_path = '/content/drive/MyDrive/AI_Projects/smoking_hackathon/'
train_path = base_path + 'data/train.csv'
test_path = base_path + 'data/test.csv'
submission_path = base_path + 'data/sample_submission.csv'
result_path = base_path + 'results/'

In [ ]:
# 1-4. 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
set_seed(42)

print("✅ 라이브러리 임포트 완료!")

## 📌 STEP 2: 데이터 로드 및 분석

In [ ]:
# 2-1. 데이터 로드
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print("=" * 50)
print("📊 데이터 기본 정보")
print("=" * 50)
print(f"Train: {train.shape}")
print(f"Test: {test.shape}")
print(f"\n컬럼: {train.columns.tolist()}")

# 타겟 분포 (중요!)
print(f"\n🎯 타겟 분포:")
print(train['label'].value_counts())
smoking_ratio = train['label'].mean()
print(f"\n흡연자 비율: {smoking_ratio*100:.2f}%")
print(f"비흡연자 비율: {(1-smoking_ratio)*100:.2f}%")

In [ ]:
# 2-2. 흡연자 vs 비흡연자 특성 차이 분석 (피처 엔지니어링 힌트)
print("\n📊 흡연자 vs 비흡연자 평균 차이:")
print("=" * 50)

numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c.lower() not in ['id', 'label']]

comparison = train.groupby('label')[numeric_cols].mean().T
comparison.columns = ['비흡연(0)', '흡연(1)']
comparison['차이'] = comparison['흡연(1)'] - comparison['비흡연(0)']
comparison['차이율(%)'] = (comparison['차이'] / (comparison['비흡연(0)'] + 0.001) * 100).round(2)
print(comparison.sort_values('차이율(%)', ascending=False))

## 📌 STEP 3: 데이터 전처리

In [ ]:
# 3-1. 원본 복사 및 ID 처리
train_df = train.copy()
test_df = test.copy()

# ID 저장
if 'ID' in test_df.columns:
    test_id = test_df['ID'].copy()
elif 'id' in test_df.columns:
    test_id = test_df['id'].copy()
else:
    test_id = pd.Series(range(len(test_df)))

# ID 제거
train_df = train_df.drop(['ID', 'id'], axis=1, errors='ignore')
test_df = test_df.drop(['ID', 'id'], axis=1, errors='ignore')

# 특성/타겟 분리
X = train_df.drop('label', axis=1, errors='ignore')
y = train_df['label']
X_test = test_df.drop('label', axis=1, errors='ignore')

feature_cols = X.columns.tolist()
print(f"원본 특성 수: {len(feature_cols)}")

## 📌 STEP 4: 피처 엔지니어링 (V3 강화!) ⭐

In [ ]:
def create_features_v3(df):
    """
    V3 피처 엔지니어링 - 대폭 강화 버전
    """
    df = df.copy()
    
    # 컬럼명 소문자 매핑
    col_map = {c: c.lower() for c in df.columns}
    df_l = df.rename(columns=col_map)
    cols = df_l.columns.tolist()
    
    # ============================================
    # 1. 콜레스테롤 관련 (흡연자 HDL↓, LDL↑)
    # ============================================
    if 'hdl' in cols and 'ldl' in cols:
        df['HDL_LDL_ratio'] = df_l['hdl'] / (df_l['ldl'] + 1)
        df['LDL_HDL_ratio'] = df_l['ldl'] / (df_l['hdl'] + 1)
        df['LDL_HDL_diff'] = df_l['ldl'] - df_l['hdl']
    
    if 'cholesterol' in cols and 'hdl' in cols:
        df['HDL_Chol_ratio'] = df_l['hdl'] / (df_l['cholesterol'] + 1)
        df['NonHDL_Chol'] = df_l['cholesterol'] - df_l['hdl']  # Non-HDL 콜레스테롤
        df['Atherogenic_idx'] = (df_l['cholesterol'] - df_l['hdl']) / (df_l['hdl'] + 1)
    
    if 'triglyceride' in cols and 'hdl' in cols:
        df['TG_HDL_ratio'] = df_l['triglyceride'] / (df_l['hdl'] + 1)  # 인슐린 저항성
    
    if 'triglyceride' in cols and 'cholesterol' in cols:
        df['TG_Chol_ratio'] = df_l['triglyceride'] / (df_l['cholesterol'] + 1)
    
    # ============================================
    # 2. 간 기능 (흡연자 GTP↑, AST/ALT 변화)
    # ============================================
    if 'gtp' in cols:
        df['GTP_log'] = np.log1p(df_l['gtp'])
        df['GTP_sqrt'] = np.sqrt(df_l['gtp'])
        df['GTP_sq'] = df_l['gtp'] ** 2
        df['GTP_high'] = (df_l['gtp'] > 50).astype(int)  # 높은 GTP 플래그
    
    if 'ast' in cols and 'alt' in cols:
        df['AST_ALT_ratio'] = df_l['ast'] / (df_l['alt'] + 1)
        df['Liver_sum'] = df_l['ast'] + df_l['alt']
        df['Liver_diff'] = abs(df_l['ast'] - df_l['alt'])
    
    if 'gtp' in cols and 'ast' in cols:
        df['GTP_AST_ratio'] = df_l['gtp'] / (df_l['ast'] + 1)
    
    # ============================================
    # 3. 헤모글로빈 (흡연자 높음 - 산소 보상)
    # ============================================
    if 'hemoglobin' in cols:
        df['Hemo_sq'] = df_l['hemoglobin'] ** 2
        df['Hemo_log'] = np.log1p(df_l['hemoglobin'])
        df['Hemo_high'] = (df_l['hemoglobin'] > 15).astype(int)  # 높은 헤모글로빈 플래그
        df['Hemo_vhigh'] = (df_l['hemoglobin'] > 16).astype(int)  # 매우 높음
    
    # 헤모글로빈 × 다른 특성 (상호작용)
    if 'hemoglobin' in cols and 'gtp' in cols:
        df['Hemo_x_GTP'] = df_l['hemoglobin'] * df_l['gtp']
    
    if 'hemoglobin' in cols and 'triglyceride' in cols:
        df['Hemo_x_TG'] = df_l['hemoglobin'] * df_l['triglyceride']
    
    # ============================================
    # 4. 혈압 관련
    # ============================================
    if 'systolic' in cols and 'diastolic' in cols:
        df['Pulse_pressure'] = df_l['systolic'] - df_l['diastolic']
        df['MAP'] = df_l['diastolic'] + (df_l['systolic'] - df_l['diastolic']) / 3
        df['BP_ratio'] = df_l['systolic'] / (df_l['diastolic'] + 1)
        df['BP_high'] = ((df_l['systolic'] > 130) | (df_l['diastolic'] > 85)).astype(int)
    
    # ============================================
    # 5. 체형 관련
    # ============================================
    if 'height' in cols and 'weight' in cols:
        height_m = df_l['height'] / 100
        df['BMI_calc'] = df_l['weight'] / (height_m ** 2 + 0.01)
        df['BMI_sq'] = df['BMI_calc'] ** 2
    
    if 'waist' in cols:
        df['Waist_high'] = (df_l['waist'] > 90).astype(int)  # 복부비만
        if 'height' in cols:
            df['Waist_Height'] = df_l['waist'] / (df_l['height'] + 1)
    
    # ============================================
    # 6. 시력/청력
    # ============================================
    eye_cols = [c for c in cols if 'eyesight' in c]
    if len(eye_cols) >= 2:
        df['Eyesight_avg'] = df_l[eye_cols].mean(axis=1)
        df['Eyesight_diff'] = abs(df_l[eye_cols[0]] - df_l[eye_cols[1]])
        df['Eyesight_min'] = df_l[eye_cols].min(axis=1)
    
    hear_cols = [c for c in cols if 'hearing' in c]
    if len(hear_cols) >= 2:
        df['Hearing_sum'] = df_l[hear_cols].sum(axis=1)
        df['Hearing_problem'] = (df_l[hear_cols].sum(axis=1) > 2).astype(int)
    
    # ============================================
    # 7. 나이 관련 (누적 효과)
    # ============================================
    if 'age' in cols:
        age = df_l['age']
        df['Age_sq'] = age ** 2
        df['Age_group'] = pd.cut(age, bins=[0,30,40,50,60,100], labels=[0,1,2,3,4]).astype(float)
        
        # 나이 × 주요 특성
        if 'hemoglobin' in cols:
            df['Age_x_Hemo'] = age * df_l['hemoglobin']
        if 'gtp' in cols:
            df['Age_x_GTP'] = age * df_l['gtp']
        if 'triglyceride' in cols:
            df['Age_x_TG'] = age * df_l['triglyceride']
        if 'systolic' in cols:
            df['Age_x_SBP'] = age * df_l['systolic']
        if 'cholesterol' in cols:
            df['Age_x_Chol'] = age * df_l['cholesterol']
    
    # ============================================
    # 8. 혈당 관련
    # ============================================
    fbs_cols = [c for c in cols if 'blood' in c or 'fasting' in c or 'sugar' in c or 'glucose' in c]
    if len(fbs_cols) > 0:
        fbs = df_l[fbs_cols[0]]
        df['FBS_log'] = np.log1p(fbs)
        df['FBS_high'] = (fbs > 100).astype(int)
        df['FBS_diabetes'] = (fbs > 126).astype(int)
    
    # ============================================
    # 9. 중성지방 관련
    # ============================================
    if 'triglyceride' in cols:
        df['TG_log'] = np.log1p(df_l['triglyceride'])
        df['TG_sq'] = df_l['triglyceride'] ** 2
        df['TG_high'] = (df_l['triglyceride'] > 150).astype(int)
    
    # ============================================
    # 10. 크레아티닌 (신장 기능)
    # ============================================
    if 'serum_creatinine' in cols:
        df['Creat_log'] = np.log1p(df_l['serum_creatinine'])
        df['Creat_high'] = (df_l['serum_creatinine'] > 1.2).astype(int)
    
    # ============================================
    # 11. 종합 건강 점수
    # ============================================
    health_cols = [c for c in cols if c in ['systolic','diastolic','hemoglobin',
                                             'triglyceride','cholesterol','hdl','ldl','gtp']]
    if len(health_cols) >= 3:
        df['Health_mean'] = df_l[health_cols].mean(axis=1)
        df['Health_std'] = df_l[health_cols].std(axis=1)
        df['Health_max'] = df_l[health_cols].max(axis=1)
        df['Health_min'] = df_l[health_cols].min(axis=1)
        df['Health_range'] = df['Health_max'] - df['Health_min']
    
    # ============================================
    # 12. 흡연 위험 점수 (종합)
    # ============================================
    risk_score = 0
    if 'Hemo_high' in df.columns:
        risk_score = risk_score + df['Hemo_high']
    if 'GTP_high' in df.columns:
        risk_score = risk_score + df['GTP_high']
    if 'TG_high' in df.columns:
        risk_score = risk_score + df['TG_high']
    if 'BP_high' in df.columns:
        risk_score = risk_score + df['BP_high']
    df['Smoking_risk_score'] = risk_score
    
    # 결측치/무한값 처리
    df = df.fillna(0)
    df = df.replace([np.inf, -np.inf], 0)
    
    return df

# 피처 엔지니어링 적용
print("🔧 피처 엔지니어링 적용 중...")
X_fe = create_features_v3(X)
X_test_fe = create_features_v3(X_test)

print(f"\n✅ 피처 엔지니어링 완료!")
print(f"원본: {len(feature_cols)}개 → 새로운: {X_fe.shape[1]}개")
print(f"추가된 특성 수: {X_fe.shape[1] - len(feature_cols)}개")

## 📌 STEP 5: 전처리 (이상치, 스케일링)

In [ ]:
# 5-1. 이상치 클리핑
def clip_outliers(df, multiplier=3.0):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64', 'float32', 'int32']:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            df[col] = df[col].clip(lower=Q1-multiplier*IQR, upper=Q3+multiplier*IQR)
    return df

X_clipped = clip_outliers(X_fe)
X_test_clipped = clip_outliers(X_test_fe)

# 5-2. Train/Val 분할
X_train, X_val, y_train, y_val = train_test_split(
    X_clipped, y, test_size=0.2, random_state=42, stratify=y
)

# 5-3. 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test_clipped)

# 전체 데이터용
scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X_clipped)
X_test_final = scaler_full.transform(X_test_clipped)

print(f"Train: {X_train_scaled.shape}")
print(f"Val: {X_val_scaled.shape}")
print(f"Test: {X_test_final.shape}")
print("\n✅ 전처리 완료!")

## 📌 STEP 6: 모델 튜닝 (⭐ Accuracy 기준!)

In [ ]:
# 6-1. XGBoost 튜닝 (Accuracy 기준 + class_weight)
print("🔧 XGBoost 튜닝 중... (Accuracy 기준)")

# 클래스 가중치 계산
n_neg = (y == 0).sum()
n_pos = (y == 1).sum()
scale_pos_weight = n_neg / n_pos
print(f"클래스 불균형 비율: {scale_pos_weight:.2f}")

xgb_params = {
    'n_estimators': [300, 500, 700, 1000],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'min_child_weight': [1, 3, 5, 7],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [1, 2, 5],
    'scale_pos_weight': [1, scale_pos_weight]  # 클래스 가중치
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss'),
    xgb_params,
    n_iter=100,  # 더 많이!
    cv=5,
    scoring='accuracy',  # ⭐ Accuracy 기준!
    random_state=42,
    n_jobs=-1,
    verbose=1
)
xgb_search.fit(X_full_scaled, y)
print(f"\n✅ XGBoost 최고 Accuracy: {xgb_search.best_score_:.5f}")

In [ ]:
# 6-2. LightGBM 튜닝
print("🔧 LightGBM 튜닝 중... (Accuracy 기준)")

lgb_params = {
    'n_estimators': [300, 500, 700, 1000],
    'max_depth': [3, 5, 7, -1],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'num_leaves': [15, 31, 63],
    'min_child_samples': [10, 20, 30, 50],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0, 0.1, 0.5],
    'class_weight': ['balanced', None]  # 클래스 가중치
}

lgb_search = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    lgb_params,
    n_iter=100,
    cv=5,
    scoring='accuracy',  # ⭐ Accuracy 기준!
    random_state=42,
    n_jobs=-1,
    verbose=1
)
lgb_search.fit(X_full_scaled, y)
print(f"\n✅ LightGBM 최고 Accuracy: {lgb_search.best_score_:.5f}")

In [ ]:
# 6-3. CatBoost 튜닝
print("🔧 CatBoost 튜닝 중... (Accuracy 기준)")

cat_params = {
    'n_estimators': [300, 500, 700, 1000],
    'max_depth': [4, 5, 6, 7],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'l2_leaf_reg': [1, 3, 5, 7],
    'auto_class_weights': ['Balanced', None]  # 클래스 가중치
}

cat_search = RandomizedSearchCV(
    CatBoostClassifier(random_state=42, verbose=0),
    cat_params,
    n_iter=60,
    cv=5,
    scoring='accuracy',  # ⭐ Accuracy 기준!
    random_state=42,
    n_jobs=-1,
    verbose=1
)
cat_search.fit(X_full_scaled, y)
print(f"\n✅ CatBoost 최고 Accuracy: {cat_search.best_score_:.5f}")

In [ ]:
# 6-4. Random Forest 튜닝
print("🔧 Random Forest 튜닝 중... (Accuracy 기준)")

rf_params = {
    'n_estimators': [300, 500, 700],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', 'balanced_subsample', None]  # 클래스 가중치
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params,
    n_iter=60,
    cv=5,
    scoring='accuracy',  # ⭐ Accuracy 기준!
    random_state=42,
    n_jobs=-1,
    verbose=1
)
rf_search.fit(X_full_scaled, y)
print(f"\n✅ Random Forest 최고 Accuracy: {rf_search.best_score_:.5f}")

In [ ]:
# 6-5. 튜닝 결과 요약
print("\n" + "=" * 50)
print("📊 튜닝 결과 요약 (Accuracy 기준)")
print("=" * 50)
print(f"XGBoost:       {xgb_search.best_score_:.5f}")
print(f"LightGBM:      {lgb_search.best_score_:.5f}")
print(f"CatBoost:      {cat_search.best_score_:.5f}")
print(f"Random Forest: {rf_search.best_score_:.5f}")

## 📌 STEP 7: 10시드 앙상블 + 최적 임계값 탐색 ⭐

In [ ]:
# 7-1. 10개 시드로 앙상블
print("=" * 50)
print("🎯 10시드 앙상블 학습")
print("=" * 50)

seeds = [42, 123, 456, 789, 1004, 2024, 7777, 8888, 9999, 1234]

best_xgb_params = xgb_search.best_params_
best_lgb_params = lgb_search.best_params_
best_cat_params = cat_search.best_params_
best_rf_params = rf_search.best_params_

# Validation 예측 수집
val_preds = {'xgb': [], 'lgb': [], 'cat': [], 'rf': []}

for i, seed in enumerate(seeds):
    print(f"Seed {seed} ({i+1}/10) 학습 중...")
    
    # XGBoost
    xgb_m = XGBClassifier(**best_xgb_params, random_state=seed, verbosity=0, use_label_encoder=False, eval_metric='logloss')
    xgb_m.fit(X_train_scaled, y_train)
    val_preds['xgb'].append(xgb_m.predict_proba(X_val_scaled)[:, 1])
    
    # LightGBM
    lgb_m = LGBMClassifier(**best_lgb_params, random_state=seed, verbose=-1)
    lgb_m.fit(X_train_scaled, y_train)
    val_preds['lgb'].append(lgb_m.predict_proba(X_val_scaled)[:, 1])
    
    # CatBoost
    cat_m = CatBoostClassifier(**best_cat_params, random_state=seed, verbose=0)
    cat_m.fit(X_train_scaled, y_train)
    val_preds['cat'].append(cat_m.predict_proba(X_val_scaled)[:, 1])
    
    # Random Forest
    rf_m = RandomForestClassifier(**best_rf_params, random_state=seed, n_jobs=-1)
    rf_m.fit(X_train_scaled, y_train)
    val_preds['rf'].append(rf_m.predict_proba(X_val_scaled)[:, 1])

# 각 모델 평균
pred_xgb = np.mean(val_preds['xgb'], axis=0)
pred_lgb = np.mean(val_preds['lgb'], axis=0)
pred_cat = np.mean(val_preds['cat'], axis=0)
pred_rf = np.mean(val_preds['rf'], axis=0)

print("\n✅ 10시드 학습 완료!")

In [ ]:
# 7-2. 최적 가중치 탐색 (0.05 단위)
print("\n🔍 최적 가중치 탐색 중...")

best_weight_score = 0
best_weights = None

for w1 in np.arange(0.1, 0.6, 0.05):
    for w2 in np.arange(0.1, 0.6, 0.05):
        for w3 in np.arange(0.1, 0.6, 0.05):
            w4 = round(1 - w1 - w2 - w3, 2)
            if w4 >= 0.05:
                prob = w1*pred_xgb + w2*pred_lgb + w3*pred_cat + w4*pred_rf
                pred_label = (prob >= 0.5).astype(int)
                acc = accuracy_score(y_val, pred_label)
                if acc > best_weight_score:
                    best_weight_score = acc
                    best_weights = (w1, w2, w3, w4)

print(f"\n🏆 최적 가중치:")
print(f"   XGBoost: {best_weights[0]:.2f}")
print(f"   LightGBM: {best_weights[1]:.2f}")
print(f"   CatBoost: {best_weights[2]:.2f}")
print(f"   RF: {best_weights[3]:.2f}")

In [ ]:
# 7-3. 최적 임계값 탐색 (⭐ 0.01 단위로 세밀하게!)
print("\n" + "=" * 50)
print("🔍 최적 임계값(Threshold) 탐색 (0.01 단위)")
print("=" * 50)

w1, w2, w3, w4 = best_weights
val_prob = w1*pred_xgb + w2*pred_lgb + w3*pred_cat + w4*pred_rf

best_threshold = 0.5
best_acc = 0
best_f1 = 0

results = []

# 0.01 단위로 세밀하게!
for thresh in np.arange(0.25, 0.65, 0.01):
    pred_label = (val_prob >= thresh).astype(int)
    acc = accuracy_score(y_val, pred_label)
    f1 = f1_score(y_val, pred_label)
    
    results.append({'threshold': thresh, 'accuracy': acc, 'f1': f1})
    
    if acc > best_acc:
        best_acc = acc
        best_f1 = f1
        best_threshold = thresh

# 결과 출력
results_df = pd.DataFrame(results)
print("\nThreshold별 성능 (상위 10개):")
print(results_df.nlargest(10, 'accuracy').to_string(index=False))

print(f"\n🏆 최적 임계값: {best_threshold:.2f}")
print(f"   Accuracy: {best_acc:.5f}")
print(f"   F1-Score: {best_f1:.5f}")

In [ ]:
# 7-4. 혼동 행렬 확인
val_pred_final = (val_prob >= best_threshold).astype(int)

print("\n📊 Validation 혼동 행렬:")
print(confusion_matrix(y_val, val_pred_final))
print("\n📊 Classification Report:")
print(classification_report(y_val, val_pred_final, target_names=['비흡연(0)', '흡연(1)']))

## 📌 STEP 8: 최종 예측

In [ ]:
# 8-1. 전체 데이터로 최종 학습
print("=" * 50)
print("📝 최종 모델 학습 (전체 데이터)")
print("=" * 50)

final_preds = {'xgb': [], 'lgb': [], 'cat': [], 'rf': []}

for i, seed in enumerate(seeds):
    print(f"Seed {seed} ({i+1}/10) 최종 학습 중...")
    
    xgb_m = XGBClassifier(**best_xgb_params, random_state=seed, verbosity=0, use_label_encoder=False, eval_metric='logloss')
    xgb_m.fit(X_full_scaled, y)
    final_preds['xgb'].append(xgb_m.predict_proba(X_test_final)[:, 1])
    
    lgb_m = LGBMClassifier(**best_lgb_params, random_state=seed, verbose=-1)
    lgb_m.fit(X_full_scaled, y)
    final_preds['lgb'].append(lgb_m.predict_proba(X_test_final)[:, 1])
    
    cat_m = CatBoostClassifier(**best_cat_params, random_state=seed, verbose=0)
    cat_m.fit(X_full_scaled, y)
    final_preds['cat'].append(cat_m.predict_proba(X_test_final)[:, 1])
    
    rf_m = RandomForestClassifier(**best_rf_params, random_state=seed, n_jobs=-1)
    rf_m.fit(X_full_scaled, y)
    final_preds['rf'].append(rf_m.predict_proba(X_test_final)[:, 1])

print("\n✅ 최종 학습 완료!")

In [ ]:
# 8-2. 앙상블 + 임계값 적용
test_xgb = np.mean(final_preds['xgb'], axis=0)
test_lgb = np.mean(final_preds['lgb'], axis=0)
test_cat = np.mean(final_preds['cat'], axis=0)
test_rf = np.mean(final_preds['rf'], axis=0)

# 가중 평균
test_prob = w1*test_xgb + w2*test_lgb + w3*test_cat + w4*test_rf

# ⭐ 최적 임계값 적용하여 0/1 변환
final_prediction = (test_prob >= best_threshold).astype(int)

print(f"🎯 사용된 임계값: {best_threshold:.2f}")
print(f"\n예측 결과 분포:")
print(f"   0 (비흡연): {(final_prediction == 0).sum()}명 ({(final_prediction == 0).mean()*100:.1f}%)")
print(f"   1 (흡연):   {(final_prediction == 1).sum()}명 ({(final_prediction == 1).mean()*100:.1f}%)")

## 📌 STEP 9: 제출 파일 생성

In [ ]:
# 9-1. 제출 파일 생성
submission_df = submission.copy()
submission_df['label'] = final_prediction
submission_df['label'] = submission_df['label'].astype(int)

print("📋 제출 파일 미리보기:")
display(submission_df.head(10))

print(f"\nlabel 데이터 타입: {submission_df['label'].dtype}")
print(f"label 고유값: {sorted(submission_df['label'].unique())}")

In [ ]:
# 9-2. 저장
output_path = result_path + 'submission_v3_final.csv'
submission_df.to_csv(output_path, index=False)
print(f"✅ 저장 완료: {output_path}")

In [ ]:
# 9-3. 검증
print("\n🔍 제출 파일 최종 검증:")
print(f"   행 개수: {len(submission_df)}")
print(f"   컬럼: {submission_df.columns.tolist()}")
print(f"   label 타입: {submission_df['label'].dtype}")
print(f"   label 값: {sorted(submission_df['label'].unique())}")
print(f"   결측치: {submission_df['label'].isnull().sum()}")

if submission_df['label'].dtype in ['int64','int32'] and set(submission_df['label'].unique()).issubset({0,1}):
    print("\n✅ 검증 통과! 제출 가능합니다.")
else:
    print("\n⚠️ 검증 실패!")

## 📌 STEP 10: 다운로드

In [ ]:
# 10-1. 다운로드
from google.colab import files
files.download(output_path)

print("\n" + "=" * 60)
print("🎉 V3 최종 버전 완료!")
print("=" * 60)
print(f"\n📥 다운로드된 파일: submission_v3_final.csv")
print(f"\n📊 설정값:")
print(f"   - 최적 임계값: {best_threshold:.2f}")
print(f"   - 가중치: XGB={w1:.2f}, LGB={w2:.2f}, CAT={w3:.2f}, RF={w4:.2f}")
print(f"   - 시드 수: {len(seeds)}개")
print(f"   - 피처 수: {X_fe.shape[1]}개")
print(f"\n📊 Validation 성능:")
print(f"   - Accuracy: {best_acc:.5f} ({best_acc*100:.2f}%)")
print(f"   - F1-Score: {best_f1:.5f}")
print(f"\n📊 예측 분포:")
print(f"   - 비흡연(0): {(final_prediction==0).sum()}명")
print(f"   - 흡연(1): {(final_prediction==1).sum()}명")
print("\n🚀 이 파일을 해커톤에 제출하세요!")
print("\n행운을 빕니다! 🍀")